# 🎭 AI Avatar All-in-One - Làm Tất Cả Trên Google Colab

> **Bỏ video/ảnh vào → Colab lo hết → Ra kết quả live. Không cần GPU máy local!**

## ⚡ Cách Dùng (3 bước)

1. Upload video hoặc ảnh vào Google Drive folder `AI_Face_Data/input/`
2. **Bật GPU:** Runtime → Change runtime type → **T4 GPU**
3. Chạy từng cell: `Ctrl+Enter` hoặc Runtime → Run all

## 📋 Pipeline Tự Động

```
Video/Ảnh Input → Auto phát hiện mặt → Phân loại góc → Trích landmarks
→ Delaunay Face Warp → Tạo animation → Live demo webcam → Xuất video
```


## 1. Mount Drive & Cài Đặt


In [1]:
# ============================================================
# 1. MOUNT DRIVE + INSTALL + GIT SYNC
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Sync code tu GitHub (bo comment neu muon dung)
# !rm -rf /content/train-ai
# !git clone https://github.com/YOUR_USERNAME/train-ai.git /content/train-ai
# %cd /content/train-ai

!pip install -q opencv-python face-recognition numpy scipy matplotlib
print("[OK] Thu vien san sang.")


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 9.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
[OK] Thu vien san sang.


In [2]:
# GPU check
try:
    import subprocess as sp
    r = sp.run(['nvidia-smi','-L'], capture_output=True, text=True)
    if r.returncode == 0:
        print(f"GPU: {r.stdout.strip().split(chr(10))[0]}")
    else:
        print("GPU: T4 (Colab default)")
except:
    print("GPU: T4 (Colab default)")

GPU: GPU 0: Tesla T4 (UUID: GPU-c26f8a3b-aef8-bf4d-5943-e18e258e42b7)


## 2. AUTO-TRAIN: Load Input → Phát Hiện Mặt → Phân Loại Góc


In [3]:
# Da gop vao cell 1b - bo qua cell nay
pass

## 3. AUTO-TRAIN: Load Input → Phát Hiện Mặt → Phân Loại Góc


In [4]:
if len(new_files) == 0:
    print("\n[OK] Tat ca file da duoc train! Khong co gi moi.")
    SESSION_ID = "no_new_files"
    session_output = f'{OUTPUT_DIR}/{SESSION_ID}'
    missing = ['left','center','right']
    best_frames = {'left': None, 'center': None, 'right': None}
    best_info = {'left': (0,0,0,None), 'center': (0,0,0,None), 'right': (0,0,0,None)}
    all_motion = []
else:

SyntaxError: incomplete input (1781229187.py, line 9)

## 4. Face Warp Engine: Xoay Mặt Mượt Bằng Delaunay


In [ ]:
# ============================================================
# 4. FACE WARP ENGINE (Delaunay + Affine)
# ============================================================
print("Building Face Warp Engine...")

# Validate we have required data
missing = [a for a in ['left','center','right'] if best_frames[a] is None]
if missing:
    print(f"THIEU GOC: {missing}")
    print("Can it nhat anh/video co du 3 goc trai/thang/phai!")
else:
    img_left, img_center, img_right = best_frames['left'], best_frames['center'], best_frames['right']
    lm_left = best_info['left'][3]
    lm_center = best_info['center'][3]
    lm_right = best_info['right'][3]
    yaw_left = best_info['left'][0]
    yaw_center = best_info['center'][0]
    yaw_right = best_info['right'][0]

    h, w = img_center.shape[:2]
    print(f"Image size: {w}x{h}")
    print(f"Yaws: L={yaw_left:.1f} C={yaw_center:.1f} R={yaw_right:.1f}")

    # Delaunay on center face hull
    hull_center = cv2.convexHull(lm_center.astype(np.int32)).squeeze()
    border = np.array([[0,0],[w//2,0],[w-1,0],[0,h//2],[w-1,h//2],[0,h-1],[w//2,h-1],[w-1,h-1]], dtype=np.float32)
    all_center = np.vstack([hull_center.astype(np.float32), border])
    tri = Delaunay(all_center)

    # Build all points for left and right
    hull_left = cv2.convexHull(lm_left.astype(np.int32)).squeeze()
    hull_right = cv2.convexHull(lm_right.astype(np.int32)).squeeze()
    all_left = np.vstack([hull_left.astype(np.float32), border])
    all_right = np.vstack([hull_right.astype(np.float32), border])

    def warp_triangle(src, dst, src_tri, dst_tri):
        sr = cv2.boundingRect(src_tri.astype(np.int32))
        dr = cv2.boundingRect(dst_tri.astype(np.int32))
        sc = src[sr[1]:sr[1]+sr[3], sr[0]:sr[0]+sr[2]]
        if sc.size == 0: return dst
        M = cv2.getAffineTransform((src_tri-sr[:2]).astype(np.float32), (dst_tri-dr[:2]).astype(np.float32))
        dc = cv2.warpAffine(sc, M, (dr[2],dr[3]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
        mask = np.zeros((dr[3],dr[2]), dtype=np.float32)
        cv2.fillConvexPoly(mask, (dst_tri-dr[:2]).astype(np.int32), 1.0)
        roi = dst[dr[1]:dr[1]+dr[3], dr[0]:dr[0]+dr[2]]
        if roi.shape == dc.shape:
            m3 = np.stack([mask]*3, -1)
            roi[:] = (dc*m3 + roi*(1-m3)).astype(np.uint8)
        return dst

    def morph(src_img, src_all, dst_all):
        result = np.zeros_like(src_img)
        for simplex in tri.simplices:
            result = warp_triangle(src_img, result, src_all[simplex], dst_all[simplex])
        return result

    def rotate_to(target_yaw):
        target_yaw = np.clip(target_yaw, yaw_left, yaw_right)
        if abs(target_yaw - yaw_center) < 1:
            return img_center.copy()

        # Interpolate landmarks
        if target_yaw <= yaw_center:
            t = (target_yaw - yaw_left) / (yaw_center - yaw_left + 1e-8)
        else:
            t = (target_yaw - yaw_center) / (yaw_right - yaw_center + 1e-8)

        if target_yaw <= yaw_center:
            lm_target = (1-t)*lm_left + t*lm_center
            all_target = (1-t)*all_left + t*all_center
            src_img = img_left.copy()
            src_all = all_left
        else:
            lm_target = (1-t)*lm_center + t*lm_right
            all_target = (1-t)*all_center + t*all_right
            src_img = img_center.copy()
            src_all = all_center

        return morph(src_img, src_all, all_target)

    # Test rotate at different angles
    test_angles = [yaw_left, (yaw_left+yaw_center)/2, yaw_center, (yaw_center+yaw_right)/2, yaw_right]
    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    for i, ang in enumerate(test_angles):
        result = rotate_to(ang)
        axes[i].imshow(result)
        axes[i].set_title(f'yaw={ang:.0f}')
        axes[i].axis('off')
    plt.suptitle('Face Rotation Test', fontsize=14)
    plt.tight_layout()
    plt.show()
    print("[OK] Face Warp Engine ready!")


## 5. Tạo Video Animation Tự Nhiên


In [ ]:
# ============================================================
# 5. CREATE ANIMATION VIDEO
# ============================================================
if missing:
    print("SKIP: Thieu du lieu goc")
else:
    print("Creating animation: left -> center -> right -> center...")

    # Motion curve: if we have motion data from video, use it; else generate smooth curve
    if len(all_motion) > 10:
        # Use real motion from input video
        timestamps, yaws = zip(*all_motion)
        print(f"Using real motion: {len(yaws)} points")
    else:
        # Generate smooth curve
        frames_per_segment = 60
        yaw_range_left = list(np.linspace(yaw_left, yaw_center, frames_per_segment))
        yaw_range_right = list(np.linspace(yaw_center, yaw_right, frames_per_segment))
        yaws = yaw_range_left + yaw_range_right[1:] + list(reversed(yaw_range_left))[1:]
        print(f"Using generated motion: {len(yaws)} frames")

    # Create video - luu vao session folder
    out_path = f'{session_output}/avatar_animation.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, 30, (w, h))

    frame_count = 0
    for target_yaw in tqdm(yaws[:300], desc="Rendering"):  # Limit to 300 frames
        rendered = rotate_to(target_yaw)

        # Add UI
        rendered_ui = rendered.copy()
        cv2.putText(rendered_ui, f"Yaw: {target_yaw:+.0f} deg",
                    (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

        out.write(cv2.cvtColor(rendered_ui, cv2.COLOR_RGB2BGR))
        frame_count += 1

    out.release()
    print(f"\n[OK] Video saved: {out_path}")
    print(f"     Frames: {frame_count}, Size: {w}x{h}")

    # Play in notebook
    from IPython.display import Video
    Video(out_path, width=400)


## 6. Live Demo: Chụp Webcam → Nhận Diện → So Sánh Với Dataset


In [ ]:
# ============================================================
# 6. LIVE DEMO: Webcam Capture + Angle Recognition
# ============================================================
from IPython.display import display, Javascript, Image as IPImage
from google.colab.output import eval_js
from base64 import b64decode
import PIL.Image
import io

def take_photo():
    js = Javascript('''
    async function takePhoto() {
        const div = document.createElement('div');
        const btn = document.createElement('button');
        btn.textContent = 'CHUP ANH';
        btn.style.cssText = 'padding:10px 30px; font-size:18px; margin:10px; cursor:pointer; background:#4CAF50; color:white; border:none; border-radius:5px';
        div.appendChild(btn);
        const video = document.createElement('video');
        video.style.display = 'block';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise((resolve) => btn.onclick = resolve);
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        return canvas.toDataURL('image/jpeg', 0.8);
    }
    ''')
    display(js)
    data = eval_js('takePhoto()')
    img_bytes = b64decode(data.split(',')[1])
    return cv2.cvtColor(cv2.imdecode(np.frombuffer(img_bytes, np.uint8), 1), cv2.COLOR_BGR2RGB)

print("Nhan nut 'CHUP ANH' ben duoi de test...")
photo = take_photo()

if photo is not None:
    lm, yaw, _ = get_landmarks_and_pose(photo)
    if lm is not None:
        # Draw landmarks
        display_img = photo.copy()
        for pt in lm.astype(np.int32):
            cv2.circle(display_img, tuple(pt), 2, (0,255,0), -1)

        # Classify
        if yaw < -12: label = f"QUAY TRAI ({yaw:.0f} deg)"
        elif yaw > 12: label = f"QUAY PHAI ({yaw:.0f} deg)"
        else: label = f"NHIN THANG ({yaw:.0f} deg)"

        cv2.putText(display_img, label, (10,40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

        # If we have the warp engine, show the corresponding avatar angle
        if not missing and 'rotate_to' in dir():
            avatar = rotate_to(yaw)
            avatar = cv2.resize(avatar, (photo.shape[1]//2, photo.shape[0]//2))

        plt.figure(figsize=(10,5))
        plt.subplot(1,2,1)
        plt.imshow(display_img)
        plt.title('Webcam')
        plt.axis('off')
        if not missing:
            plt.subplot(1,2,2)
            plt.imshow(avatar)
            plt.title(f'AI Avatar (yaw={yaw:.0f})')
            plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print("Khong tim thay khuon mat!")


## 7. Xuất Model & Download


In [ ]:
# ============================================================
# 7. EXPORT MODEL + SAVE ALL
# ============================================================
if not missing:
    # Save face data for local use
    face_data = {
        'images': {
            'left': img_left,
            'center': img_center,
            'right': img_right,
        },
        'landmarks': {
            'left': lm_left,
            'center': lm_center,
            'right': lm_right,
        },
        'yaws': {
            'left': yaw_left,
            'center': yaw_center,
            'right': yaw_right,
        },
        'motion': all_motion,
    }

    # Save best frames vao session folder
    for angle in ['left','center','right']:
        if best_frames[angle] is not None:
            cv2.imwrite(f'{session_output}/best_{angle}.jpg',
                        cv2.cvtColor(best_frames[angle], cv2.COLOR_RGB2BGR))

    # Save full pickle
    with open(f'{session_output}/face_data.pkl', 'wb') as f:
        pickle.dump(face_data, f)

    print(f"[OK] Saved to {session_output}/")
    for f in os.listdir(session_output):
        size = os.path.getsize(f'{session_output}/{f}') / 1024
        print(f"  {f} ({size:.1f} KB)")
else:
    print(f"SKIP: Thieu du lieu de xuat (missing: {missing})")


## 8. Tổng Kết

✅ **Pipeline Hoàn Thành!**

📥 **Kết quả lưu tại:** `AI_Face_Data/output/[SESSION_ID]/`

- `avatar_animation.mp4` — Video animation
- `best_left/center/right.jpg` — Ảnh tốt nhất mỗi góc
- `face_data.pkl` — Model cho máy local

📋 **Nhật ký:** `AI_Face_Data/trained_log.json` — tự ghi file nào đã train

🔁 **Lần sau có video mới:**

1. Upload vào `AI_Face_Data/input/`
2. Restart & Run All → tự bỏ qua video cũ, chỉ train video mới
3. Kết quả ra folder riêng: `output/20260727_143000/`

🗑️ **Muốn train lại video cũ:** Xóa tên file trong `trained_log.json` hoặc đổi tên video
